In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# Input data files are available in the read-only "../input/" directory
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# CTD Python Data Pipeline: Global Superstore

**Author:** Brian Bazurto (GitHub: ba-00001)
**Course:** Code the Dream, Python 100 (final project / capstone)

## Question and goal

The Global Superstore dataset has 51,290 order line items from a large retailer that sells furniture, office supplies and technology in 147 countries (2011 to 2014). I study business analytics and supply chain, so I want to use it to answer one main question:

> **What actually drives profit, as opposed to just sales?**

The sub-questions I am working toward:

1. **Discounts:** how much do discounts hurt profit margin, and is there a point where they turn orders into losses?
2. **Products:** which categories and sub-categories make money and which lose it?
3. **Geography:** which markets and regions are the most and least profitable?
4. **Shipping:** how do ship mode and order priority relate to shipping time and cost?

## Dataset

- **Source:** [Global Superstore on Kaggle](https://www.kaggle.com/datasets/anandaramg/global-superstore/data) (uploaded by anandaramg), file `Global Superstore.txt`.
- **Format:** tab-separated text, UTF-8, 51,290 rows x 27 columns.
- **Grain:** one row per product line within an order (`Row ID`). An `Order ID` can have several rows.
- **Columns:** order and ship dates, ship mode, order priority, customer and segment, location (city / state / country / market / region), product and category, and the numeric fields `Sales`, `Quantity`, `Discount`, `Profit`, `Shipping Cost`. It also has a few helper columns (`Year`, `weeknum`, `Market2` and a record-count column) that I look at in the cleaning section.

## How this notebook is built

This project is built incrementally, one section per week:

| Week | Section | Status |
|---|---|---|
| 5 | Loading, inspection and data cleaning | Done |
| 6 | Data wrangling, aggregation and feature engineering | Done |
| 7+ | Visualization, analysis and conclusions | Coming later |

## Setup

Imports and display settings. Everything used here comes preinstalled on Kaggle (pandas, numpy, matplotlib).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  # used for the plots in Week 7+

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

print("pandas", pd.__version__, "| numpy", np.__version__)

---
# Week 5: Loading, Inspection and Cleaning

## 1. Loading the data

The Kaggle input path depends on how the dataset was added (on my run it was `/kaggle/input/datasets/anandaramg/global-superstore/Global Superstore.txt`), so I do not hard-code it. The function below walks `/kaggle/input` and picks up any file with "superstore" in the name and a `.txt`, `.csv` or `.xlsx` extension. If the notebook runs outside Kaggle it falls back to a local `./data/` folder.

The Kaggle file is a **tab-separated** `.txt`, so it needs `sep="\t"`. With the default comma separator everything would end up in one column. The loader chooses the separator from the extension and also handles a comma CSV or an Excel copy of the same data.

In [ ]:
SEARCH_DIRS = ["/kaggle/input", "./data", "."]
EXT_PRIORITY = [".txt", ".csv", ".xlsx"]  # prefer the original Kaggle .txt


def find_data_file(search_dirs=SEARCH_DIRS, keyword="superstore"):
    """Return the first data file whose name contains `keyword` (searched recursively)."""
    for base in search_dirs:
        if not os.path.isdir(base):
            continue
        matches = []
        for root, _, files in os.walk(base):
            for f in files:
                ext = Path(f).suffix.lower()
                if keyword in f.lower() and ext in EXT_PRIORITY:
                    matches.append(Path(root) / f)
        if matches:
            return sorted(matches, key=lambda p: EXT_PRIORITY.index(p.suffix.lower()))[0]
    raise FileNotFoundError("Global Superstore file not found. On Kaggle, use Add Input to attach the dataset.")


def load_superstore(path):
    """Load the dataset: .txt is tab-separated, .csv comma-separated, .xlsx Excel."""
    path = Path(path)
    ext = path.suffix.lower()
    if ext == ".xlsx":
        return pd.read_excel(path)
    sep = "\t" if ext == ".txt" else ","
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(path, sep=sep, encoding=enc)
        except UnicodeDecodeError:
            print(f"{enc} failed, trying next encoding...")
    raise ValueError(f"Could not decode {path}")


data_path = find_data_file()
raw = load_superstore(data_path)
print("Loaded:", data_path)
print("Shape:", raw.shape)
print("Columns:", list(raw.columns))

## 2. First look at the data

`head()` and `tail()` to see what rows look like, `info()` for types and non-null counts, and `describe()` for the numeric ranges.

In [ ]:
raw.head()

In [ ]:
raw.tail()

In [ ]:
raw.info()

In [ ]:
raw.describe().T

In [ ]:
# summary of the text columns: how many unique values each one has
raw.describe(include="object").T

### What I noticed

- **Column order is alphabetical** (`Category`, `City`, `Country`, ...), so related fields such as the two dates are far apart. I reorder them during cleaning so the table reads like an order record.
- **Extra helper columns.** There is a column named `记录数` (Chinese for "record count"), plus `Year`, `weeknum` and `Market2`. They look like leftovers from a BI/pivot tool. I need to check whether they add information or just repeat other columns.
- **Dates are strings.** `Order Date` and `Ship Date` load as `object` with values like `2011-01-07 00:00:00.000`. They have to be real datetimes before I can compute shipping time or group by month.
- **`Sales` is an integer.** Sales are rounded to whole dollars, while `Profit` and `Shipping Cost` keep their decimals. `Profit` goes negative, which is expected (loss-making orders).
- **`Customer ID` vs `Customer Name`:** there are far more IDs than names, so the ID is probably not one-per-person.
- **`Sales` and `Profit` are very skewed**: the max is many times the median. I need to decide whether those are outliers to remove or real big orders.

Each check below is followed by the fix, and each fix is a function, so at the end `clean_superstore()` runs the whole cleaning process from the raw file in one call.

## 3. Constant and redundant columns

In [ ]:
# columns with only one value carry no information
nunique = raw.nunique()
constant_cols = list(nunique[nunique == 1].index)
print("Constant columns:", constant_cols)
for c in constant_cols:
    print(f"  {c!r} -> only value: {raw[c].unique()}")

In [ ]:
# Market2 vs Market: does each Market map to exactly one Market2?
print("Market values:", raw["Market"].nunique(), "| Market2 values:", raw["Market2"].nunique())
print("Markets mapping to more than one Market2:",
      (raw.groupby("Market")["Market2"].nunique() > 1).sum())
pd.crosstab(raw["Market"], raw["Market2"])

**Finding:**
- `记录数` has the value `1` on every row. It is a row counter for pivot tables, and `len()` or `count()` gives me the same thing. **Dropped.**
- `Market2` is just `Market` with `US` and `Canada` merged into "North America". Every `Market` maps to exactly one `Market2`, so it adds nothing I can't rebuild with a `map()`. I keep the more detailed `Market` (7 values) and **drop** `Market2`.
- `Year` and `weeknum` should come from `Order Date`. I check that after converting the dates (next step) before dropping them.

In [ ]:
def drop_constant_columns(df):
    """Drop columns that have a single value on every row."""
    nunique = df.nunique()
    const = list(nunique[nunique == 1].index)
    return df.drop(columns=const)


def drop_redundant_columns(df, cols=("Market2", "Year", "weeknum")):
    """Drop helper columns that can be rebuilt from other columns."""
    return df.drop(columns=[c for c in cols if c in df.columns])

## 4. Data types: converting the dates

The date strings look like `2011-01-07 00:00:00.000`. I give `to_datetime` the exact format so nothing is guessed, and use `errors="coerce"` so a bad value turns into `NaT` that I can count instead of breaking the notebook. If an Excel copy is loaded and the dates are already datetimes, they are left alone.

In [ ]:
DATE_COLS = ["Order Date", "Ship Date"]
DATE_FORMAT = "%Y-%m-%d %H:%M:%S.%f"


def convert_dates(df, cols=DATE_COLS, fmt=DATE_FORMAT):
    """Parse the date strings into datetime64 and report anything unparseable."""
    df = df.copy()
    for col in cols:
        if not pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = pd.to_datetime(df[col].astype(str).str.strip(), format=fmt, errors="coerce")
        bad = df[col].isna().sum()
        if bad:
            print(f"Warning: {bad} unparseable values in {col}")
    return df


dates = convert_dates(raw[DATE_COLS + ["Year", "weeknum"]])
print(dates.dtypes)
print("NaT values:", dates[DATE_COLS].isna().sum().to_dict())
print("Order Date range:", dates["Order Date"].min().date(), "to", dates["Order Date"].max().date())

Now I can check the helper columns `Year` and `weeknum` against `Order Date`. "Week number" can be defined more than one way, so I test two common ones: the spreadsheet `WEEKNUM` style (week 1 contains January 1, weeks start on Sunday) and the ISO week.

In [ ]:
od = dates["Order Date"]
print("Year mismatches vs Order Date year:", (dates["Year"] != od.dt.year).sum())

# spreadsheet-style WEEKNUM: week 1 contains Jan 1, weeks start on Sunday
jan1 = pd.to_datetime(od.dt.year.astype(str) + "-01-01")
weeknum_sheet = (od.dt.dayofyear - 1 + (jan1.dt.dayofweek + 1) % 7) // 7 + 1
weeknum_iso = od.dt.isocalendar().week.astype(int)

print("weeknum range:", dates["weeknum"].min(), "-", dates["weeknum"].max())
print("Match rate, spreadsheet WEEKNUM: {:.1%}".format((dates["weeknum"] == weeknum_sheet).mean()))
print("Match rate, ISO week:            {:.1%}".format((dates["weeknum"] == weeknum_iso).mean()))

**Finding:** `Year` matches the order date's year on every row, so it is fully redundant. The output above shows which week definition `weeknum` follows. Either way it is derived from `Order Date`, and I can recompute any date part I need with the `.dt` accessor (which I do in Week 6). Both are **dropped** to avoid having two sources of truth for the same fact.

I also convert the text columns with only a few repeated values to `category`, and make `Order Priority` an **ordered** category (Low < Medium < High < Critical) so it sorts correctly later.

In [ ]:
CATEGORY_COLS = ["Ship Mode", "Segment", "Market", "Region", "Category", "Sub-Category"]
PRIORITY_ORDER = ["Low", "Medium", "High", "Critical"]


def convert_categories(df):
    """Low-cardinality text columns to category; priority as ordered category."""
    df = df.copy()
    for col in CATEGORY_COLS:
        df[col] = df[col].astype("category")
    df["Order Priority"] = pd.Categorical(df["Order Priority"], categories=PRIORITY_ORDER, ordered=True)
    return df

## 5. Missing values

In [ ]:
missing = raw.isna().sum()
print("Total missing values:", missing.sum())
missing[missing > 0]

**Finding:** the file has **no missing values** in any column, so there is nothing to fill or drop.

I still add a guard to the pipeline: rows missing any *key* field (IDs, dates, sales, profit) get dropped with `dropna(subset=...)`, because those rows cannot be used in any aggregation. On this file it removes 0 rows, but it protects the pipeline if a later version of the data has gaps. Descriptive fields like city or state would be filled with `"Unknown"` instead of dropping the whole row.

Missing values *created* by my own calculations are handled where they come up. For example, profit margin is undefined when `Sales` is 0 (see the range checks below and Week 6).

In [ ]:
KEY_COLS = ["Order ID", "Order Date", "Ship Date", "Customer ID", "Product ID", "Sales", "Profit", "Quantity"]
TEXT_FILL_COLS = ["City", "State", "Country", "Customer Name", "Product Name"]


def handle_missing(df):
    """Drop rows missing key fields; fill descriptive text fields with 'Unknown'."""
    before = len(df)
    df = df.dropna(subset=[c for c in KEY_COLS if c in df.columns])
    print(f"Rows dropped for missing key fields: {before - len(df)}")
    fill = {c: "Unknown" for c in TEXT_FILL_COLS if c in df.columns}
    return df.fillna(fill)

## 6. Duplicates

First I check for exact duplicate rows. `Row ID` is a unique counter, so I check both with it and without it (two rows that are identical apart from `Row ID` would still be duplicates). Then I check a business key: the same product, in the same order, shipped the same day, in the same quantity.

In [ ]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Duplicate Row IDs:", raw["Row ID"].duplicated().sum())
print("Duplicates ignoring Row ID:", raw.drop(columns="Row ID").duplicated().sum())

DUP_KEY = ["Order ID", "Product ID", "Ship Date", "Quantity"]
print("Duplicates by", DUP_KEY, ":", raw.duplicated(subset=DUP_KEY).sum())

In [ ]:
# look at the key-based duplicates side by side: which columns differ?
key_dupes = raw[raw.duplicated(subset=DUP_KEY, keep=False)].sort_values(DUP_KEY + ["Row ID"])
differs = key_dupes.groupby(DUP_KEY).nunique().gt(1).sum()
print("Columns that differ inside a duplicate group:\n", differs[differs > 0])
key_dupes[["Row ID", "Order ID", "Product ID", "Quantity", "Sales", "Profit", "Shipping Cost"]].head(6)

**Finding:** there are **no exact duplicates**, not even after ignoring `Row ID`. 9 rows repeat another row on order, product, ship date and quantity. The output above shows that inside those pairs the `Row ID` and `Shipping Cost` differ.

**Decision:** I keep `drop_duplicates()` (ignoring `Row ID`) in the pipeline, so exact repeats in a future version of the file would be removed. I **keep** the 9 near-duplicates. Their shipping costs differ, which looks like a real split shipment rather than a row copied by mistake, and 9 rows out of 51,290 (0.02%) cannot move any aggregate in a meaningful way.

In [ ]:
def remove_duplicates(df):
    """Drop exact duplicates, ignoring the Row ID counter."""
    before = len(df)
    df = df.drop_duplicates(subset=[c for c in df.columns if c != "Row ID"])
    print(f"Duplicates removed: {before - len(df)}")
    return df

## 7. Text columns: whitespace, casing and IDs

### Whitespace and casing

Stray spaces or different casing would make one value look like two different groups in a `groupby`. So I check every text column.

In [ ]:
text_cols = raw.select_dtypes(include="object").columns.drop(DATE_COLS)
ws = {c: int((raw[c] != raw[c].str.strip()).sum()) for c in text_cols}
print("Values with leading/trailing whitespace:", sum(ws.values()))

for c in ["Ship Mode", "Segment", "Market", "Region", "Category", "Sub-Category", "Order Priority"]:
    if raw[c].nunique() != raw[c].str.lower().nunique():
        print("Case conflict in", c)
print("Ship Mode values:", sorted(raw["Ship Mode"].unique()))
print("Market values:", sorted(raw["Market"].unique()))
print("Order Priority values:", sorted(raw["Order Priority"].unique()))

**Finding:** no whitespace problems and no casing conflicts, and the category labels are consistent (`US`, `EU`, `APAC`, ...). I keep a `normalize_strings()` step in the pipeline anyway: it strips spaces and collapses double spaces. On this file it changes nothing, but it is cheap and protects the group-bys.

In [ ]:
def normalize_strings(df):
    """Strip whitespace and collapse repeated spaces in every text column."""
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
    return df

### Customer ID vs Customer Name

In [ ]:
print("Unique Customer IDs:  ", raw["Customer ID"].nunique())
print("Unique Customer Names:", raw["Customer Name"].nunique())
print("IDs linked to more than one name:", (raw.groupby("Customer ID")["Customer Name"].nunique() > 1).sum())

ids_per_name = raw.groupby("Customer Name")["Customer ID"].nunique()
print("\nIDs per customer name:")
print(ids_per_name.describe()[["mean", "min", "max"]])

# the IDs for one customer and the markets they appear in
example = ids_per_name.idxmax()
raw.loc[raw["Customer Name"] == example].groupby(["Customer ID", "Market"]).size().rename("lines").head(10)

**Finding:** there are 4,873 customer IDs but only 795 names. No ID belongs to more than one name, but one name has several IDs. The IDs end in an extra digit that changes with the market (for example `LS-172304`), so a `Customer ID` really means "this customer in this market".

**Decision:** I keep both columns as they are. For **customer-level** analysis (like lifetime value in Week 6) I group by `Customer Name`, because grouping by ID would split one person into several "customers". Limitation: if two different real people shared a name they would be merged. I accept that because splitting one person into several market-level "customers" would distort lifetime value a lot more.

### Product ID vs Product Name

In [ ]:
names_per_pid = raw.groupby("Product ID")["Product Name"].nunique()
multi = names_per_pid[names_per_pid > 1]
print("Product IDs:", raw["Product ID"].nunique(), "| Product names:", raw["Product Name"].nunique())
print("Product IDs with more than one name:", len(multi))

# example: one ID and the different names it is used for
pid = multi.index[0]
raw.loc[raw["Product ID"] == pid, ["Product ID", "Product Name", "Sub-Category"]].drop_duplicates()

**Finding:** 457 product IDs are used for more than one product name. In the example above, a single ID covers different items. This is a quirk of how the source data was built, and I can't tell which name is "right".

**Decision:** I do not change the data. I avoid grouping by `Product ID` and use `Product Name` (or `Sub-Category`) as the product label in the analysis.

## 8. Discount values

Retail discounts normally come in round steps, so I list every distinct discount value to look for anything odd.

In [ ]:
disc_counts = raw["Discount"].value_counts().sort_index()
print("Distinct discount values:", len(disc_counts))
# show 3 decimals so values like 0.002 are not hidden by the 2-decimal display setting
print(disc_counts.rename(index="{:.3f}".format).to_string())

In [ ]:
# values ending in ...02 (0.002, 0.202, 0.402, 0.602): where do they come from?
odd = raw["Discount"].round(3).isin([0.002, 0.202, 0.402, 0.602])
print("Rows with an x.x02 discount:", odd.sum())
raw.loc[odd].groupby(["Market", "Discount"]).size().unstack(fill_value=0)

**Finding:** there are 27 distinct discounts between 0 and 0.85. Most are normal steps (0.1, 0.2, 0.5 ...), and some markets use steps like 0.07 / 0.17 / 0.27. A few values look off by a tiny amount: **0.002, 0.202, 0.402, 0.602**. The output above shows which markets they appear in. They read like 0%, 20%, 40% and 60% with a small rounding or tax artifact added.

**Decision:** I keep the raw values because I can't prove what the "true" discount was, and the difference is 0.2 percentage points. When I put discounts into bands in Week 6, I choose bin edges so that 0.202 falls in the same band as 0.20, 0.402 with 0.40, and so on. That way the artifact does not push a row into a higher discount band.

## 9. Validating ranges and checking outliers

Before deciding on outliers I check that the values are at least *possible*: no negative sales, quantity at least 1, discount between 0 and 1, shipping cost not negative, and ship date not before order date.

In [ ]:
checks = {
    "Sales < 0": (raw["Sales"] < 0).sum(),
    "Sales == 0": (raw["Sales"] == 0).sum(),
    "Profit < 0": (raw["Profit"] < 0).sum(),
    "Profit == 0": (raw["Profit"] == 0).sum(),
    "Quantity < 1": (raw["Quantity"] < 1).sum(),
    "Discount outside 0-1": (~raw["Discount"].between(0, 1)).sum(),
    "Shipping Cost < 0": (raw["Shipping Cost"] < 0).sum(),
    "Ship before order": (dates["Ship Date"] < dates["Order Date"]).sum(),
}
ship_days = (dates["Ship Date"] - dates["Order Date"]).dt.days
print({k: int(v) for k, v in checks.items()})
print("Days between order and ship: min", ship_days.min(), "| max", ship_days.max(), "| mean", round(ship_days.mean(), 2))

In [ ]:
# the row(s) with zero sales
raw.loc[raw["Sales"] == 0, ["Order ID", "Product Name", "Quantity", "Discount", "Sales", "Profit"]]

**Finding:** nothing is impossible. There are no negative sales or quantities, every discount is between 0 and 0.85, and every ship date falls 0 to 7 days after its order date. Negative profit (12,544 rows) is a real business outcome, not an error. It is what I want to study.

There is **one row with `Sales` = 0**. `Sales` is rounded to whole dollars, so this is a very cheap item that rounded down to $0. I keep the row because its quantity and profit are still real, but profit margin (`Profit / Sales`) is undefined for it. In Week 6 its margin is set to `NaN` instead of dividing by zero. The rounding also means per-line margins on very small sales are approximate, so I compute group margins from **totals** (sum of profit / sum of sales).

In [ ]:
def iqr_outliers(s, k=1.5):
    """Boolean mask of values outside the IQR fences."""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)


num_cols = ["Sales", "Quantity", "Discount", "Profit", "Shipping Cost"]
outlier_summary = pd.DataFrame({
    "iqr_outliers": [iqr_outliers(raw[c]).sum() for c in num_cols],
    "pct_of_rows": [iqr_outliers(raw[c]).mean() * 100 for c in num_cols],
    "median": [raw[c].median() for c in num_cols],
    "max": [raw[c].max() for c in num_cols],
}, index=num_cols)
outlier_summary

In [ ]:
# what do the biggest sales look like?
raw.nlargest(5, "Sales")[["Order ID", "Country", "Sub-Category", "Product Name", "Quantity", "Discount", "Sales", "Profit"]]

**Finding:** the IQR rule flags a lot of rows: 5,655 for `Sales`, 9,755 for `Profit` (about 19%), 5,909 for `Shipping Cost`, 4,172 for `Discount` and 877 for `Quantity`. That is just the shape of retail data: most line items are small and a few are big.

**Decision: keep the outliers.** The largest sales are things like copiers, phones, machines and furniture bought in quantity. They are real orders, not typos (all range checks passed). They are also the lines behind the biggest profits *and* the biggest losses, which is exactly what my question is about, so removing them would hide the most important part of the story. For later charts I will use medians or log scales where the skew gets in the way.

## 10. Running the cleaning pipeline

All the cleaning steps chained into one function, so the whole process can be rerun from the raw file in one call. The last step puts the columns in a logical order (IDs, dates, shipping, customer, location, product, numbers).

In [ ]:
COLUMN_ORDER = ["Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Order Priority",
                "Customer ID", "Customer Name", "Segment", "City", "State", "Country", "Market", "Region",
                "Product ID", "Category", "Sub-Category", "Product Name",
                "Sales", "Quantity", "Discount", "Profit", "Shipping Cost"]


def reorder_columns(df, order=COLUMN_ORDER):
    """Put known columns in a logical order; anything else goes at the end."""
    first = [c for c in order if c in df.columns]
    return df[first + [c for c in df.columns if c not in first]]


def clean_superstore(df):
    """Full Week 5 cleaning pipeline: raw dataframe in, clean dataframe out."""
    df = drop_constant_columns(df)
    df = drop_redundant_columns(df)
    df = normalize_strings(df)
    df = handle_missing(df)
    df = remove_duplicates(df)
    df = convert_dates(df)
    df = convert_categories(df)
    df = reorder_columns(df)
    return df.reset_index(drop=True)


df = clean_superstore(raw)
df.info()

In [ ]:
# quick validation of the cleaned data
assert df["Order Date"].notna().all() and df["Ship Date"].notna().all()
assert df.isna().sum().sum() == 0
assert not {"Market2", "Year", "weeknum"} & set(df.columns)
print("Columns: {} -> {}".format(raw.shape[1], df.shape[1]))
print("Date range:", df["Order Date"].min().date(), "to", df["Order Date"].max().date())
print("Rows:", len(df), "| Orders:", df["Order ID"].nunique(),
      "| Customer IDs:", df["Customer ID"].nunique(), "| Customer names:", df["Customer Name"].nunique())
df.head()

### Week 5 cleaning summary

| Issue | What I found | What I did |
|---|---|---|
| File format | Tab-separated `.txt`, columns in alphabetical order | Loader uses `sep="\t"` for `.txt`; columns reordered logically |
| Constant column | `记录数` ("record count") is 1 on every row | Dropped (generic constant-column check) |
| Redundant columns | `Market2` = `Market` with US + Canada merged; `Year` and `weeknum` derived from `Order Date` | Validated against `Market` / `Order Date`, then dropped |
| Date types | Dates stored as `YYYY-MM-DD 00:00:00.000` strings | `pd.to_datetime` with an explicit format, verified 0 `NaT` |
| Categories | Repeated text labels; priority has a natural order | Converted to `category`; `Order Priority` is an ordered category |
| Missing values | None in the file | Guard in pipeline: `dropna` on key fields, `fillna("Unknown")` on descriptive text |
| Duplicates | 0 exact duplicates; 9 key-based near-duplicates differing in shipping cost | `drop_duplicates()` in the pipeline; near-duplicates kept as split shipments |
| Whitespace / casing | None found | `normalize_strings()` kept as a safety step |
| Customer IDs | 4,873 IDs vs 795 names (one ID per customer per market) | Kept; customer-level analysis groups by `Customer Name` |
| Product IDs | 457 IDs used for more than one product name | Kept; analysis uses `Product Name` / `Sub-Category`, not `Product ID` |
| Discount values | Artifacts like 0.002 / 0.202 / 0.402 / 0.602 | Kept raw; Week 6 bins built so they group with 0 / 0.2 / 0.4 / 0.6 |
| Zero sales | 1 row with `Sales` = 0 (rounded to whole dollars) | Kept; its margin will be `NaN`, and group margins come from totals |
| Outliers | Heavy right skew in `Sales` / `Profit` | Range-validated; kept as real large orders |

---
## Coming in Week 6+
Wrangling, aggregations and engineered features (Week 6), then visualizations and conclusions (Week 7+).